# Multi-Dataset Insurance Benchmark: TabPFN vs GLM

Runs the same TabPFN vs GLM comparison across **four public insurance classification datasets**
to test whether TabPFN v2.6 consistently outperforms a tuned logistic regression.

| # | Dataset | Rows | Features | Target | Pos. Rate |
|---|---------|------|----------|--------|----------|
| 1 | eudirectlapse (EU direct insurer) | 23,060 | 18 | lapse | 12.8% |
| 2 | coil2000 (Dutch insurer, COIL 2000, OpenML 298) | 9,822 | 85 | CARAVAN | 6.0% |
| 3 | ausprivauto0405 (AU vehicle 2004-05, CASdatasets) | 67,856 | 6 | ClaimOcc | 6.8% |
| 4 | freMTPL2freq_binary (FR MTPL 50K sample) | 50,000 | 10 | ClaimIndicator | 5.0% |

**Design:**
- All models capped at 10,000 training samples (stratified) for a fair comparison.
- 20% hold-out test set (stratified).
- Backend: `tabpfn_client` (cloud inference).
- Metrics: ROC AUC and PR AUC.

Run `python scripts/download_datasets.py` first if data files are missing.


In [ ]:
import sys, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.utils.class_weight import compute_class_weight
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

REPO_ROOT        = Path('../../')
DATA_DIR         = REPO_ROOT / 'data' / 'raw'
OUT_DIR          = REPO_ROOT / 'data' / 'processed'
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED      = 42
GLOBAL_MAX_TRAIN = 10_000   # all models capped here for fair comparison
TEST_SIZE        = 0.20

print('Data dir:', DATA_DIR.resolve())
print('Files:   ', sorted(p.name for p in DATA_DIR.iterdir()))

In [ ]:
from IPython.display import Markdown, display

backend = 'client'   # 'client' uses tabpfn_client cloud API; 'local' uses local tabpfn

if backend == 'local':
    from tabpfn import TabPFNClassifier
    display(Markdown('**Backend:** tabpfn (local)'))
elif backend == 'client':
    from tabpfn_client import TabPFNClassifier, init
    try:
        init(use_server=True)
        display(Markdown('**Backend:** tabpfn_client (cloud API) — authenticated'))
    except Exception as e:
        display(Markdown(f'tabpfn_client init: {e}'))
else:
    raise ValueError(f'Unknown backend: {backend!r}')

print(f'backend={backend!r}')

In [ ]:
# (csv_filename, target_column_name, display_label)
DATASETS = [
    ('eudirectlapse.csv',       'lapse',           'EU Direct Lapse'),
    ('coil2000.csv',            'CARAVAN',         'COIL 2000 (NL)'),
    ('ausprivauto0405.csv',     'ClaimOcc',        'Aus. Vehicle (AU)'),
    ('freMTPL2freq_binary.csv', 'ClaimIndicator',  'freMTPL2 Binary (FR)'),
]

for fname, _, label in DATASETS:
    p = DATA_DIR / fname
    status = 'OK     ' if p.exists() else 'MISSING  (run: python scripts/download_datasets.py)'
    print(f'  {status}  {label:<28} {fname}')

In [ ]:
def build_preprocessor(X, scale=True):
    """Auto-detect numeric/categorical columns; optionally apply StandardScaler."""
    num_cols = X.select_dtypes(include=['number']).columns.tolist()
    cat_cols = X.select_dtypes(exclude=['number']).columns.tolist()
    transformers = []
    if num_cols:
        num_steps = [('imp', SimpleImputer(strategy='median'))]
        if scale:
            num_steps.append(('scl', StandardScaler()))
        transformers.append(('num', Pipeline(num_steps), num_cols))
    if cat_cols:
        transformers.append(('cat', Pipeline([
            ('imp', SimpleImputer(strategy='most_frequent')),
            ('enc', OrdinalEncoder(
                handle_unknown='use_encoded_value', unknown_value=-1)),
        ]), cat_cols))
    return ColumnTransformer(transformers, remainder='drop')

print('build_preprocessor defined')

In [ ]:
all_results  = []
dataset_meta = []

EQ = '=' * 60
for fname, target_col, label in DATASETS:
    print(f'\n{EQ}')
    print(f'Dataset: {label}  ({fname})')
    print(EQ)

    # Load
    df     = pd.read_csv(DATA_DIR / fname)
    X_full = df.drop(columns=[target_col])
    y_full = df[target_col].astype(int)
    pos    = y_full.mean()
    print(f'  Rows: {len(df):,}  Features: {X_full.shape[1]}  +rate: {pos:.1%}')
    dataset_meta.append({'name': label, 'rows': len(df),
                         'features': X_full.shape[1], 'positive_rate': pos})

    # Train/test split
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_full, y_full, test_size=TEST_SIZE,
        random_state=RANDOM_SEED, stratify=y_full)

    # Cap training size
    if len(X_tr) > GLOBAL_MAX_TRAIN:
        X_tr, _, y_tr, _ = train_test_split(
            X_tr, y_tr, train_size=GLOBAL_MAX_TRAIN,
            random_state=RANDOM_SEED, stratify=y_tr)

    print(f'  Train: {len(X_tr):,}  Test: {len(X_te):,}')

    # Class weights
    cw   = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_tr.values)
    cw_d = {0: cw[0], 1: cw[1]}

    # Preprocessors
    prep_skl = build_preprocessor(X_tr, scale=True)   # scaled  -> LogReg, RF
    prep_tab = build_preprocessor(X_tr, scale=False)  # unscaled -> TabPFN, tree models

    X_tr_skl = prep_skl.fit_transform(X_tr)
    X_te_skl = prep_skl.transform(X_te)
    X_tr_tab = prep_tab.fit_transform(X_tr)
    X_te_tab = prep_tab.transform(X_te)
    y_tr_a, y_te_a = y_tr.values, y_te.values

    # sklearn / tree models
    sklearn_models = {
        'LogisticRegression': LogisticRegression(
            max_iter=1000, class_weight='balanced', random_state=RANDOM_SEED),
        'RandomForest': RandomForestClassifier(
            n_estimators=200, class_weight='balanced',
            random_state=RANDOM_SEED, n_jobs=-1),
        'CatBoost': CatBoostClassifier(
            iterations=300, learning_rate=0.05, depth=6,
            class_weights=cw_d, random_seed=RANDOM_SEED, verbose=0),
        'XGBoost': XGBClassifier(
            n_estimators=300, max_depth=6, learning_rate=0.05,
            scale_pos_weight=cw_d[1] / cw_d[0],
            random_state=RANDOM_SEED, eval_metric='logloss', verbosity=0),
    }

    for mname, mdl in sklearn_models.items():
        t0    = time.time()
        mdl.fit(X_tr_skl, y_tr_a)
        proba = mdl.predict_proba(X_te_skl)[:, 1]
        secs  = time.time() - t0
        roc   = roc_auc_score(y_te_a, proba)
        pr    = average_precision_score(y_te_a, proba)
        all_results.append({'dataset': label, 'model': mname,
                             'roc_auc': roc, 'pr_auc': pr, 'time_s': secs})
        print(f'  {mname:<22} ROC={roc:.4f}  PR={pr:.4f}  ({secs:.1f}s)')

    # TabPFN
    print('  TabPFN (waiting for cloud inference)...')
    t0  = time.time()
    tab = (TabPFNClassifier(n_estimators=8, device='auto', random_state=RANDOM_SEED)
           if backend == 'local' else TabPFNClassifier(random_state=RANDOM_SEED))
    tab.fit(X_tr_tab, y_tr_a)
    proba_tab = tab.predict_proba(X_te_tab)[:, 1]
    secs      = time.time() - t0
    roc_tab   = roc_auc_score(y_te_a, proba_tab)
    pr_tab    = average_precision_score(y_te_a, proba_tab)
    all_results.append({'dataset': label, 'model': 'TabPFN',
                        'roc_auc': roc_tab, 'pr_auc': pr_tab, 'time_s': secs})
    print(f'  {"TabPFN":<22} ROC={roc_tab:.4f}  PR={pr_tab:.4f}  ({secs:.1f}s)')

print('\nAll benchmarks complete.')

In [ ]:
from IPython.display import display, Markdown

results_df    = pd.DataFrame(all_results)
model_order   = ['LogisticRegression', 'TabPFN', 'CatBoost', 'RandomForest', 'XGBoost']
dataset_order = [lbl for _, _, lbl in DATASETS]

pivot_roc = (results_df
             .pivot(index='dataset', columns='model', values='roc_auc')
             .reindex(index=dataset_order, columns=model_order)
             .round(4))

pivot_pr = (results_df
            .pivot(index='dataset', columns='model', values='pr_auc')
            .reindex(index=dataset_order, columns=model_order)
            .round(4))

def hl_max(row):
    best = row.max()
    return ['font-weight:bold; color:darkgreen' if v == best else '' for v in row]

display(Markdown('## ROC AUC by Dataset and Model'))
display(pivot_roc.style.apply(hl_max, axis=1).format('{:.4f}'))

display(Markdown('## PR AUC by Dataset and Model'))
display(pivot_pr.style.apply(hl_max, axis=1).format('{:.4f}'))

# Head-to-head TabPFN vs GLM
display(Markdown('## TabPFN vs GLM Head-to-Head (ROC AUC)'))
hth = pivot_roc[['LogisticRegression', 'TabPFN']].copy()
hth['delta (TabPFN - GLM)'] = (hth['TabPFN'] - hth['LogisticRegression']).round(4)
hth['TabPFN wins']          =  hth['TabPFN'] > hth['LogisticRegression']
display(hth)

wins = hth['TabPFN wins'].sum()
total = len(hth)
display(Markdown(f'### Summary: TabPFN beats GLM on **{wins}/{total}** datasets (ROC AUC)'))

In [ ]:
model_order   = ['LogisticRegression', 'TabPFN', 'CatBoost', 'RandomForest', 'XGBoost']
colors        = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0', '#FF9800']
x_labels      = ['GLM', 'TabPFN', 'CatBoost', 'RF', 'XGB']
dataset_order = [lbl for _, _, lbl in DATASETS]

fig, axes = plt.subplots(1, 4, figsize=(18, 5), sharey=False)
fig.suptitle('ROC AUC: TabPFN vs Baselines across Insurance Datasets',
             fontsize=12, fontweight='bold')

for ax, label in zip(axes, dataset_order):
    sub  = (results_df[results_df['dataset'] == label]
            .set_index('model').reindex(model_order))
    vals = sub['roc_auc'].values.astype(float)
    bars = ax.bar(range(len(model_order)), vals, color=colors,
                  edgecolor='black', linewidth=0.5)
    ax.set_title(label, fontsize=9, fontweight='bold')
    ax.set_xticks(range(len(model_order)))
    ax.set_xticklabels(x_labels, fontsize=8)
    ax.set_ylabel('ROC AUC')
    valid = vals[~np.isnan(vals)]
    if len(valid):
        span = max(valid.max() - valid.min(), 0.02)
        ax.set_ylim(max(0.0, valid.min() - span * 0.5),
                    min(1.0, valid.max() + span * 0.5 + 0.01))
    for bar, v in zip(bars, vals):
        if not np.isnan(v):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.001,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
out_fig = OUT_DIR / 'multi_dataset_roc_comparison.png'
plt.savefig(out_fig, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved: {out_fig}')

In [ ]:
out_csv = OUT_DIR / 'multi_dataset_benchmark_results.csv'
results_df.to_csv(out_csv, index=False)
print(f'Results saved: {out_csv}')
display(results_df)